# BlackBox — Phase 2: Generation, Conflict Resolution, Layer Scoping

This picks up where Phase 1 left off. We're using `MemoryClient` (the hosted mem0 platform
API), so this notebook reconnects to whatever Sessions 1–3 already stored — nothing is
replayed from scratch.

Three things get built here, in order:

1. **Generation** — actually answer a question using retrieved memories, not just print them
2. **Conflict resolution** — force a fact to change and watch mem0 reconcile it
3. **Layer scoping** — use `user_id` / `run_id` / `agent_id` filters the way they're meant to be used


## 0. Reconnect to the existing memory store

Same `user_id` as Phase 1 (`eng_01`). If Sessions 1–3 are still there, `get_all` below should
print the Boeing 737 / fleet / troubleshooting facts without us adding anything new yet.


In [1]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()  # pulls MEM0_API_KEY from your .env file

client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))

# Sanity check: confirm Phase 1's memories are still here before building on top of them
existing = client.get_all(filters={"user_id": "eng_01"})
for item in existing.get("results", []):
    print("•", item["memory"])

• User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
• User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
• User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects


## 1. Generation — turn retrieval into an actual answer

We also bring back `lab_llm_config.py` from Phase 1. mem0's hosted API stores and retrieves
memories, but it doesn't answer questions for us — that's still our job.


In [8]:
from lab_llm_config import complete, load_lab_env


def answer_with_memory(question, user_id):
  # Step 1: retrieve the memories mem0 thinks are relevant to this question
  results = client.search(query=question, filters={"user_id": user_id})
  memories = [r["memory"] for r in results.get("results", [])]

  # Step 2: format them into a system prompt
  memory_block = "\n".join(f"- {m}" for m in memories)
  system_prompt = (
      "You are an aircraft maintenance assistant. "
      "Use the facts below about this engineer if they're relevant. "
      "Don't mention that you're using 'stored memories' -- just answer"
      " naturally.\n\n"
      f"Known facts about this user:\n{memory_block}"
  )

  # Step 3: call the LLM completion function with system prompt + question
  full_prompt = f"{system_prompt}\n\nQuestion: {question}"
  answer = complete(full_prompt)  # <--- Changed from load_lab_env to complete

  return answer, memories


### Session 4 — the payoff moment from Phase 1, now with a real answer

In [9]:
question = "What's the backup system?"
answer, used_memories = answer_with_memory(question, user_id="eng_01")

print("Memories used:")
for m in used_memories:
    print(" -", m)

print("\nAnswer:")
print(answer)

Memories used:
 - The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
 - User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
 - User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
 - User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft

Answer:
The backup for the 737’s hydraulic system is the **Standby Hydraulic System**.  
It’s a separate, electrically‑powered pump (usually driven by the APU or an electric motor) that can take over if either System A or System B fails. The standby system is isolated from the primary systems and is used only when a primary system is out of service or during a loss‑of‑pressure event.


### Store this turn as a new memory too

Just like Phase 1's raw `.add()` calls -- this session's Q&A becomes tomorrow's context.


In [10]:
client.add(
    [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ],
    user_id="eng_01",
)
print("Session 4 turn stored.")

Session 4 turn stored.


## 2. Conflict resolution — force a fact to change

Session 5: the engineer says they've moved fleets. We snapshot memory *before* and *after*
this call so the change is visible, rather than assumed.


In [11]:
# Snapshot BEFORE -- don't print yet, just save it
before = client.get_all(filters={"user_id": "eng_01"})
before_facts = [item["memory"] for item in before.get("results", [])]

In [15]:
# The conflicting fact
add_result = client.add(
    "Actually, I've moved off the 737 -- I'm on the 787 fleet now.",
    user_id="eng_01",
)

# On v3, add() typically only reports ADD events -- it won't necessarily
# label this as an "update" even though that's effectively what happens.
# Print it anyway so you see exactly what the API hands back.
print(add_result)




{'event_id': '0cac6851-78f6-4d75-803b-8364177ced9e', 'status': 'PENDING'}


In [16]:
# Snapshot AFTER
after = client.get_all(filters={"user_id": "eng_01"})
after_facts = [item["memory"] for item in after.get("results", [])]

print("BEFORE:")
for f in before_facts:
    print(" -", f)

print("\nAFTER:")
for f in after_facts:
    print(" -", f)

BEFORE:
 - The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
 - User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
 - User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
 - User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects

AFTER:
 - User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
 - The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually driven by the APU or an electric motor, which can take over if System A or System B fails and is isolated from the primary systems, used only when a primary system is out of service or during a loss‑of‑pressure event.
 - The Boeing 737 hydraulic system includes System

### Confirm it at the retrieval level too

The real test isn't the diff above -- it's whether a *new* question correctly reflects the change.


In [17]:
results = client.search(
    query="what hydraulic system do I work on?",
    filters={"user_id": "eng_01"},
)

for r in results.get("results", []):
    print("-", r["memory"])

- User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
- User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
- The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
- The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually driven by the APU or an electric motor, which can take over if System A or System B fails and is isolated from the primary systems, used only when a primary system is out of service or during a loss‑of‑pressure event.
- User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
- User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps


Whatever these three cells actually show is the real answer -- don't assume mem0 deleted
the 737 fact outright just because the docs say it reconciles conflicts. It might keep both
and simply rank 787 higher at retrieval time. Either behavior is worth discussing with the
class; the point of this section is to look, not to guess.


## 3. Layer scoping with `user_id` / `run_id` / `agent_id`

mem0's real scoping parameters map onto the four memory layers from the original design:

| Layer | mem0 filter |
|---|---|
| User (durable) | `user_id` only |
| Session (ephemeral) | `user_id` **and** `run_id` |
| Agent (behavior rule) | `agent_id` |

### 3a. A session-scoped fact

Something that only matters to *this* troubleshooting episode, not the user's permanent record.


In [18]:
client.add(
    "Pressure gauge reading low during today's preflight check.",
    user_id="eng_01",
    run_id="session_5",
)
print("Session-scoped fact stored under run_id='session_5'.")

Session-scoped fact stored under run_id='session_5'.


### 3b. Test: does a plain `user_id` search pick up the session-scoped fact?

Don't assume the answer -- this is exactly the kind of thing to verify directly.


In [19]:
# Search WITHOUT specifying run_id
results_user_only = client.search(
    query="preflight check today",
    filters={"user_id": "eng_01"},
)
print("user_id only:")
for r in results_user_only.get("results", []):
    print(" -", r["memory"])

user_id only:
 - User observed a low hydraulic pressure gauge reading during a preflight check on a Boeing 737 on August 3, 2026
 - User observed a low hydraulic pressure gauge reading during preflight checks on a Boeing 737 and asked for troubleshooting steps
 - User is seeking guidance on how often to check hydraulic fluid levels for their fleet of Boeing 737 aircraft
 - The Boeing 737 hydraulic system includes System A and System B as primary systems and a Standby system as the backup hydraulic system
 - User has transitioned from working on Boeing 737 hydraulic systems to now working on the Boeing 787 fleet
 - User works on Boeing 737 hydraulic systems, handling their maintenance and engineering aspects
 - The backup for the Boeing 737 hydraulic system is the Standby Hydraulic System, a separate electrically‑powered pump usually driven by the APU or an electric motor, which can take over if System A or System B fails and is isolated from the primary systems, used only when a primar

In [20]:
# Search WITH run_id explicitly included
results_with_run = client.search(
    query="preflight check today",
    filters={"AND": [{"user_id": "eng_01"}, {"run_id": "session_5"}]},
)
print("user_id + run_id:")
for r in results_with_run.get("results", []):
    print(" -", r["memory"])

user_id + run_id:
 - User observed a low hydraulic pressure gauge reading during a preflight check on a Boeing 737 on August 3, 2026


### 3c. An agent-layer fact

This one isn't about the engineer at all -- it's a behavior rule for the assistant itself,
so it doesn't fit naturally into the conversation. We add it standalone to demonstrate the
mechanism honestly rather than force it into the story.

**Heads up:** there's a known open issue in mem0 where combining `user_id` and `agent_id`
filters together can return zero results even when both facts exist. Run the cells below and
see what you actually get -- treat it as a live example of a real memory-system rough edge,
not a bug in your code.


In [21]:
client.add(
    "Aviation maintenance queries need safety-critical accuracy -- always double-check specs.",
    agent_id="maintenance_agent",
)
print("Agent-layer fact stored under agent_id='maintenance_agent'.")

Agent-layer fact stored under agent_id='maintenance_agent'.


for the below cell sometimes there is not ooutput.

When using MemoryClient.add(...), memory extraction is processed asynchronously on Mem0's cloud servers:

Async Delay: client.add(..., agent_id="maintenance_agent") queues a background job. It takes 1–3 seconds for Mem0's cloud worker to process the text and write the extracted fact into storage.
Timing: If client.get_all(filters={"agent_id": "maintenance_agent"}) is run immediately in the next cell before the worker finishes, it queries the database while the resu

In [23]:
# Filter by agent_id alone
by_agent_only = client.get_all(filters={"agent_id": "maintenance_agent"})
print("agent_id only:")
for item in by_agent_only.get("results", []):
    print(" -", item["memory"])

agent_id only:
 - User emphasizes that aviation maintenance queries must have safety‑critical accuracy and that specifications should always be double‑checked.


In [24]:
# Now try combining user_id AND agent_id -- this is the case known to be flaky
by_user_and_agent = client.get_all(
    filters={"AND": [{"user_id": "eng_01"}, {"agent_id": "maintenance_agent"}]}
)
print("user_id + agent_id combined:")
for item in by_user_and_agent.get("results", []):
    print(" -", item["memory"])
print("\n(If this came back empty, that matches a known mem0 issue -- not something to debug in your own code.)")

user_id + agent_id combined:

(If this came back empty, that matches a known mem0 issue -- not something to debug in your own code.)


## Wrap-up

What Phase 2 actually demonstrated:

1. Retrieval alone isn't an assistant -- **generation** is what turns "matching facts" into
   a usable answer.
2. mem0's conflict handling is real but its exact behavior (delete vs. rerank) should be
   **observed, not assumed** -- which is exactly what the before/after diff was for.
3. `user_id`, `run_id`, and `agent_id` map cleanly onto the four-layer design from the
   architecture doc, but combining filters doesn't always behave exactly as documented --
   worth testing live in front of the class rather than promising it'll "just work."

**Possible Phase 3 direction:** decay and importance scoring -- do older, unreinforced
memories actually get deprioritized over time, or is that left entirely to the LLM's judgment
at retrieval time? Worth designing as its own experiment before writing any code.
